# SkyAI Virtual Jewellery Try-On — Rendering Pipeline (Approach C)
### 4-stage implementation, built to run on Google Colab Pro

This notebook implements the pipeline from **`SkyAI_TryOn_Pipeline_Brief.md`** exactly as specified:
a hybrid **"Approach C"** design where jewellery fidelity is a *structural* guarantee (deterministic
math, zero model calls on the piece's own pixels) rather than a *statistical* one (a diffusion model
regenerating the piece and hoping it stays faithful).

| Stage | What it does | Models |
|---|---|---|
| **1. Segmentation** | Extract a clean, alpha-matted jewellery layer | SAM2 (fine-tune plug-in noted) + ViTMatte refinement |
| **2. Anchor & Landmark Detection** | Find where/how big/what angle to place the piece | MediaPipe Face / Hand / Pose Landmarker — deterministic, no generative model |
| **3. Deterministic Placement** | Warp, scale, rotate, paste — pure math | OpenCV only, **no model calls** |
| **4. Bounded Harmonisation** | Add contact shadow / lighting integration around the piece only | FLUX.1 Fill [dev] (primary) + Qwen-Image-Edit-2511 (A/B candidate), each with an optional fine-tuned edit LoRA |

**Non-negotiable constraint carried through every stage:** shape, gem cut, proportion, metal finish,
engraving and hallmark must survive rendering unaltered. Stage 3 makes this true by construction
(no model touches the piece's pixels); Stage 4's mask — and a belt-and-braces hard re-composite of
the original pixels, implemented below — makes it true even in the one stage that *is* generative.

**Faithfulness note:** per the build brief, this notebook does not silently simplify or substitute
models. Every place a real design decision, approximation, or open question exists (e.g. per-category
landmark choices, the `arch:` key in the ai-toolkit config, the SSIM vs. LPIPS target-direction reading)
is flagged inline in a markdown or code comment at the point it matters, rather than glossed over.

**Scope note:** deployment, licensing, and infra are explicitly out of scope here (per the brief) —
this notebook is model/architecture selection and a runnable reference implementation, not a
production service.

## 1. Environment & Colab-Specific Setup

Colab's local disk does **not** survive a session reset, and Colab Pro does not guarantee a fixed
session length — so every artifact that's expensive to regenerate (model checkpoints, LoRA weights,
the training dataset) is written to a persistent Google Drive folder, not `/content`.

In [ ]:
# --- 1.1 Mount Google Drive & create the persistent project directory ---------------------------
import os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/skyai_tryon')
else:
    # Not fatal — lets this notebook still be sanity-checked outside Colab —
    # but Drive persistence (the entire point of this cell) won't apply.
    print("[WARN] google.colab not available — not running in Colab. "
          "Falling back to a local working directory; nothing here will "
          "survive a session/kernel reset.")
    PROJECT_ROOT = Path('./skyai_tryon')

CHECKPOINT_DIR         = PROJECT_ROOT / 'checkpoints'
MEDIAPIPE_MODEL_DIR    = CHECKPOINT_DIR / 'mediapipe'
LORA_OUTPUT_DIR        = PROJECT_ROOT / 'lora_output'
DATASET_DIR            = PROJECT_ROOT / 'datasets'
DATASET_REFERENCE_DIR  = DATASET_DIR / 'reference'   # Stage-3 hard-pasted composites (LoRA input)
DATASET_TARGET_DIR     = DATASET_DIR / 'target'       # retouched targets + .txt captions (LoRA target)
DEMO_ASSETS_DIR        = PROJECT_ROOT / 'demo_assets'
RUN_OUTPUTS_DIR        = PROJECT_ROOT / 'run_outputs'
AI_TOOLKIT_DIR         = PROJECT_ROOT / 'ai-toolkit'

for d in [CHECKPOINT_DIR, MEDIAPIPE_MODEL_DIR, LORA_OUTPUT_DIR, DATASET_REFERENCE_DIR,
          DATASET_TARGET_DIR, DEMO_ASSETS_DIR, RUN_OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root       : {PROJECT_ROOT.resolve()}")
print(f"Checkpoints        : {CHECKPOINT_DIR}")
print(f"LoRA output        : {LORA_OUTPUT_DIR}")
print(f"Dataset (reference): {DATASET_REFERENCE_DIR}")
print(f"Dataset (target)   : {DATASET_TARGET_DIR}")


In [ ]:
# --- 1.2 GPU / VRAM detection ---------------------------------------------------------------------
import subprocess

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout if smi.returncode == 0 else "[WARN] nvidia-smi failed — no GPU attached to this runtime "
                                              "(Runtime > Change runtime type > GPU in Colab).")

import torch

def get_vram_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

VRAM_GB = get_vram_gb()
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\nDetected device : {GPU_NAME}")
print(f"VRAM            : {VRAM_GB:.1f} GB")

MIN_VRAM_FOR_FLUX_FILL_GB = 24  # Stage 4 (FLUX.1 Fill, 12B params, bf16) is comfortable at ~24GB+

if VRAM_GB == 0:
    print("[WARN] No GPU detected. Stages 1/2/3 (SAM2, MediaPipe, OpenCV) can limp along on CPU for "
          "small images, but Stage 4 (FLUX.1 Fill / Qwen-Image-Edit) is not realistically usable "
          "without a GPU. In Colab: Runtime > Change runtime type > select a GPU.")
elif VRAM_GB < MIN_VRAM_FOR_FLUX_FILL_GB:
    print(f"[WARN] {VRAM_GB:.1f} GB VRAM is below the ~24GB Stage 4 wants. Colab Pro can hand you a "
          f"T4 (16GB) instead of an L4/A100 depending on availability at the time you connect.\n"
          f"  Fallbacks applied automatically below where possible:\n"
          f"    - bf16 loading (already the default dtype used throughout this notebook)\n"
          f"    - pipe.enable_model_cpu_offload() instead of pipe.to('cuda') for Stage-4 pipelines\n"
          f"  Manual options if that's still not enough:\n"
          f"    - fp8 quantization via `optimum-quanto` (qfloat8) on the transformer before wrapping\n"
          f"      it in the diffusers pipeline — not wired in by default since it adds a dependency\n"
          f"      and a slightly different loading path; flagged here as an extension point.\n"
          f"    - pipe.enable_sequential_cpu_offload() (slower, but the lowest-VRAM option)")
else:
    print(f"[OK] {VRAM_GB:.1f} GB VRAM — sufficient for Stage 4 in bf16 without offloading.")

USE_CPU_OFFLOAD_FALLBACK = 0 < VRAM_GB < MIN_VRAM_FOR_FLUX_FILL_GB


In [ ]:
# --- 1.3 Install dependencies ----------------------------------------------------------------------
# Pinned to versions known to work together as of this notebook's writing. `--break-system-packages`
# is used because Colab's system Python is externally-managed in newer images; harmless if not needed.
import sys, subprocess, importlib.util

def pip_install(args, label=None):
    label = label or " ".join(args)
    print(f"Installing: {label}")
    cmd = [sys.executable, "-m", "pip", "install", "--break-system-packages", "-q"] + args
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[WARN] pip install failed for: {label}\n{result.stderr[-2000:]}")

# Core CV / scientific stack
pip_install(["opencv-python-headless==4.10.0.84"])
pip_install(["mediapipe>=0.10.14"])
pip_install(["scikit-image>=0.24"])
pip_install(["lpips>=0.1.4"])
pip_install(["safetensors", "sentencepiece", "protobuf", "einops", "pyyaml", "huggingface_hub"])

# Torch is Colab-preinstalled with a CUDA-matched build — don't fight it, just ensure the
# HF ecosystem on top of it is current enough for FluxFillPipeline / QwenImageEditPlusPipeline.
pip_install(["transformers>=4.46.0", "accelerate>=0.34.0", "peft>=0.13.0"])

# diffusers: try the stable PyPI release first; fall back to GitHub main if FluxFillPipeline
# isn't in it yet (it landed in diffusers 0.31, but pin drift on a cached Colab image happens).
pip_install(["diffusers>=0.31.0"])
try:
    from diffusers import FluxFillPipeline  # noqa: F401
    print("[OK] Stable diffusers release provides FluxFillPipeline.")
except ImportError:
    print("[INFO] FluxFillPipeline missing from the resolved diffusers wheel — "
          "installing diffusers from GitHub main instead.")
    pip_install(["git+https://github.com/huggingface/diffusers.git"])

# SAM2 — official Meta repo. No stable "sam2" PyPI package as of writing; install editable from source.
if importlib.util.find_spec("sam2") is None:
    pip_install(["git+https://github.com/facebookresearch/sam2.git"], label="sam2 (from GitHub)")
else:
    print("[OK] sam2 already importable.")

print("\nDependency installation pass complete.")


In [ ]:
# --- 1.4 Resumable-training & Colab keep-alive helpers ---------------------------------------------
# A kernel *disconnect* can't be caught and handled from inside Python once it happens — Colab
# enforces a hard session ceiling regardless of Pro tier, keep-alive tricks only delay idle timeouts.
# The real resilience mechanism is: checkpoint to Drive every N steps (wired into the Stage-4 LoRA
# trainer below), and make every loader here auto-pick-up the latest Drive checkpoint on re-run.

from IPython.display import Javascript, display

def prevent_colab_idle_disconnect():
    """Best-effort: periodically clicks Colab's own 'connect' button if present, to reduce
    (not eliminate) idle-timeout disconnects during a long Stage-4 LoRA run. No-ops safely
    outside a Colab frontend."""
    try:
        display(Javascript("""
            function SkyAIKeepAlive(){
              document.querySelector("colab-connect-button")?.click();
            }
            setInterval(SkyAIKeepAlive, 60000);
        """))
        print("[OK] Keep-alive ping injected (fires every 60s while this tab stays open).")
    except Exception as e:
        print(f"[INFO] Keep-alive JS not injected (likely not a Colab frontend): {e}")

prevent_colab_idle_disconnect()

def find_latest_checkpoint(checkpoint_dir, pattern="*.safetensors"):
    """Returns the most recently modified file matching `pattern` under `checkpoint_dir`, or None.
    Used to resume LoRA training after a disconnect and to auto-load the newest LoRA for inference
    without hardcoding a step number."""
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        return None
    candidates = sorted(checkpoint_dir.rglob(pattern), key=lambda p: p.stat().st_mtime)
    return candidates[-1] if candidates else None


## 2. Stage 1 — Asset Extraction (Segmentation)

**Model:** SAM2 (`facebook/sam2-hiera-large`, or the biggest variant that fits the detected VRAM),
fine-tuned on a jewellery-specific dataset in production (plug-in point noted below — not run here,
since real annotated training data isn't available yet).

**Why fine-tune:** generic SAM2 is trained on natural photography, not macro product shots — it
clips/bleeds on thin chain links, prong settings, and faceted gem edges.

**Refinement:** a ViTMatte alpha-matting pass cleans up the jagged hairline edges a binary mask leaves
on thin chain links.

In [ ]:
# --- 2.1 Load SAM2 ------------------------------------------------------------------------------
import torch
from sam2.sam2_image_predictor import SAM2ImagePredictor

# facebook/sam2-hiera-large is what the brief specifies; drop to a smaller variant automatically
# if the detected VRAM can't comfortably hold the large checkpoint.
SAM2_REPO_IDS = {
    "large":      "facebook/sam2-hiera-large",
    "base_plus":  "facebook/sam2-hiera-base-plus",
    "small":      "facebook/sam2-hiera-small",
    "tiny":       "facebook/sam2-hiera-tiny",
}

def pick_sam2_variant(vram_gb):
    if vram_gb >= 24: return "large"
    if vram_gb >= 12: return "base_plus"
    if vram_gb >= 8:  return "small"
    return "tiny"

sam2_variant = pick_sam2_variant(VRAM_GB if VRAM_GB > 0 else 8)
sam2_repo_id = SAM2_REPO_IDS[sam2_variant]
print(f"Selected SAM2 variant: '{sam2_variant}' -> {sam2_repo_id} "
      f"(brief specifies 'large' or the biggest variant that fits detected VRAM: {VRAM_GB:.1f} GB)")

sam2_predictor = SAM2ImagePredictor.from_pretrained(sam2_repo_id, device=device)
print(f"SAM2 ({sam2_variant}) loaded on '{device}' via the Hugging Face Hub.")


In [ ]:
# --- 2.2 segment_jewellery() — brief's exact function signature ---------------------------------
import numpy as np

def segment_jewellery(image: np.ndarray, point_or_box_prompt: dict, predictor=None):
    """
    Stage 1 asset extraction.

    Args:
        image: HxWx3 uint8 RGB array of the jewellery product shot.
        point_or_box_prompt: either
            {"points": [[x, y], ...], "labels": [1, 0, ...]}  (foreground=1 / background=0 clicks), or
            {"box": [x1, y1, x2, y2]}
        predictor: a loaded SAM2ImagePredictor (defaults to the module-level `sam2_predictor`).

    Returns:
        (alpha_mask, score): HxW float32 binary-ish mask in [0, 1] from SAM2 directly (pre-matting),
        and SAM2's own confidence for the chosen mask.
    """
    predictor = predictor or sam2_predictor
    predictor.set_image(image)

    kwargs = {}
    if "points" in point_or_box_prompt:
        kwargs["point_coords"] = np.array(point_or_box_prompt["points"], dtype=np.float32)
        kwargs["point_labels"] = np.array(point_or_box_prompt["labels"], dtype=np.int32)
    if "box" in point_or_box_prompt:
        kwargs["box"] = np.array(point_or_box_prompt["box"], dtype=np.float32)
    if not kwargs:
        raise ValueError("point_or_box_prompt must contain 'points' and/or 'box'.")

    autocast_dtype = torch.bfloat16 if device == "cuda" else torch.float32
    with torch.inference_mode(), torch.autocast(device_type=device, dtype=autocast_dtype):
        masks, scores, _ = predictor.predict(multimask_output=True, **kwargs)

    best_idx = int(np.argmax(scores))
    return masks[best_idx].astype(np.float32), float(scores[best_idx])


In [ ]:
# --- 2.3 Matting refinement: SAM2 binary mask -> trimap -> clean alpha (ViTMatte, guided-filter fallback)
import cv2

def mask_to_trimap(binary_mask: np.ndarray, erode_px: int = 8, dilate_px: int = 8) -> np.ndarray:
    """Binary SAM2 mask -> 3-class trimap (0=bg, 128=unknown, 255=fg). Eroding gives the definite-
    foreground core; dilating gives the unknown boundary ring — exactly the thin-edge cleanup
    ViTMatte needs on chain links and prong settings."""
    mask_u8 = (binary_mask > 0.5).astype(np.uint8) * 255
    fg = cv2.erode(mask_u8, np.ones((erode_px, erode_px), np.uint8))
    dilated = cv2.dilate(mask_u8, np.ones((dilate_px, dilate_px), np.uint8))
    trimap = np.full_like(mask_u8, 128)
    trimap[fg > 0] = 255
    trimap[dilated == 0] = 0
    return trimap


def guided_filter_matte(image_rgb: np.ndarray, trimap: np.ndarray, radius: int = 8, eps: float = 1e-3) -> np.ndarray:
    """DOCUMENTED PLACEHOLDER — only used if a ViTMatte checkpoint can't be loaded (e.g. no internet
    access to Hugging Face at Stage-1 time). A from-scratch guided filter (no cv2.ximgproc /
    opencv-contrib dependency): treats the trimap (normalised to [0,1]) as the filtering input,
    guided by the RGB image, so the alpha edge snaps to real image gradients. This is a real
    fallback, not a stub — but it is meaningfully weaker than a trained matting network on thin
    filigree, which is why ViTMatte is the primary path."""
    guide = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    src = trimap.astype(np.float32) / 255.0

    box = lambda img, r: cv2.boxFilter(img, -1, (2 * r + 1, 2 * r + 1))
    mean_I, mean_p = box(guide, radius), box(src, radius)
    cov_Ip = box(guide * src, radius) - mean_I * mean_p
    var_I = box(guide * guide, radius) - mean_I * mean_I
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    alpha = box(a, radius) * guide + box(b, radius)
    return np.clip(alpha, 0.0, 1.0)


_vitmatte_model, _vitmatte_processor = None, None

def _load_vitmatte():
    global _vitmatte_model, _vitmatte_processor
    if _vitmatte_model is not None:
        return _vitmatte_model, _vitmatte_processor
    try:
        from transformers import VitMatteImageProcessor, VitMatteForImageMatting
        _vitmatte_processor = VitMatteImageProcessor.from_pretrained("hustvl/vitmatte-small-composition-1k")
        _vitmatte_model = VitMatteForImageMatting.from_pretrained(
            "hustvl/vitmatte-small-composition-1k"
        ).to(device).eval()
        print("[OK] ViTMatte (hustvl/vitmatte-small-composition-1k) loaded.")
    except Exception as e:
        print(f"[WARN] Could not load ViTMatte ({e}). Falling back to the guided-filter placeholder.")
        _vitmatte_model = False
    return _vitmatte_model, _vitmatte_processor


def refine_mask_with_matting(image_rgb: np.ndarray, binary_mask: np.ndarray) -> np.ndarray:
    """Stage 1 refinement: binary SAM2 mask -> trimap -> clean alpha matte. Tries ViTMatte first,
    falls back to the guided-filter placeholder if the checkpoint can't be reached."""
    trimap = mask_to_trimap(binary_mask)
    model, processor = _load_vitmatte()

    if model:
        from PIL import Image
        inputs = processor(
            images=Image.fromarray(image_rgb), trimaps=Image.fromarray(trimap), return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            alpha = model(**inputs).alphas
        alpha = alpha[0, 0].float().cpu().numpy()
        alpha = cv2.resize(alpha, (image_rgb.shape[1], image_rgb.shape[0]), interpolation=cv2.INTER_LINEAR)
        return np.clip(alpha, 0.0, 1.0)
    return guided_filter_matte(image_rgb, trimap)


def segment_jewellery_refined(image_rgb: np.ndarray, point_or_box_prompt: dict):
    """Convenience wrapper chaining 2.2 -> 2.3: returns the final RGBA asset (uint8) plus the
    float32 alpha mask, ready for Stage 3."""
    binary_mask, sam_score = segment_jewellery(image_rgb, point_or_box_prompt)
    alpha = refine_mask_with_matting(image_rgb, binary_mask)
    rgba = np.dstack([image_rgb, (alpha * 255).astype(np.uint8)])
    return rgba, alpha, sam_score


**Fine-tuning plug-in point (not run in this notebook — no annotated jewellery data available yet):**

- **Dataset format:** a few hundred–2,000 product photos across the category list below, each with
  a binary mask (or better, an alpha matte) of the piece. Standard SAM2 fine-tuning format: image +
  mask pairs, optionally with a box/point prompt recorded per image if prompts were curated rather
  than auto-sampled.
- **Where it plugs in:** replace the `SAM2ImagePredictor.from_pretrained(sam2_repo_id, ...)` call in
  2.1 with a checkpoint loaded from a fine-tuned `.pt`/safetensors file — the rest of this notebook
  (`segment_jewellery`, `refine_mask_with_matting`) needs no changes, since they only depend on the
  predictor's public `.predict()` interface.
- **LoRA vs. full fine-tune of the mask decoder:** SAM2's official fine-tuning recipe supports both.
  LoRA (rank 4–16 on the mask decoder's attention/MLP layers) is the practical default for a dataset
  this size — full fine-tuning that small a dataset risks overfitting/catastrophic forgetting of
  SAM2's general segmentation ability. Full fine-tuning becomes worth it once the jewellery dataset
  grows into the tens of thousands of images.

## 3. Stage 2 — Anchor & Landmark Detection

**Models:** MediaPipe Tasks API — `FaceLandmarker`, `HandLandmarker`, `PoseLandmarker` — one model
family per category (face mesh for earrings/maang tikka/nose ring, hands for rings/bracelets/bangles,
pose for necklaces/anklets).

**Why no generative model here:** this is a solved, deterministic landmark-detection problem.
Introducing a generative model would only add failure modes for zero benefit.

**Output payload per category function**, matching the brief's placement response payload: anchor
coordinates (normalised), a scale reference measurement, a rotation angle, an occlusion mask slot,
and a confidence score.

In [ ]:
# --- 3.1 Download MediaPipe .task model files at runtime ----------------------------------------
import urllib.request

MEDIAPIPE_MODEL_URLS = {
    "face_landmarker": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
    "hand_landmarker": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
    "pose_landmarker": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
}

mediapipe_model_paths = {}
for name, url in MEDIAPIPE_MODEL_URLS.items():
    dest = MEDIAPIPE_MODEL_DIR / f"{name}.task"
    if not dest.exists():
        print(f"Downloading {name} -> {dest}")
        urllib.request.urlretrieve(url, dest)
    else:
        print(f"[OK] {name} already cached at {dest}")
    mediapipe_model_paths[name] = str(dest)


In [ ]:
# --- 3.2 Initialise the three landmarkers --------------------------------------------------------
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

_base_opts = lambda p: mp_python.BaseOptions(model_asset_path=p)

face_landmarker = mp_vision.FaceLandmarker.create_from_options(
    mp_vision.FaceLandmarkerOptions(
        base_options=_base_opts(mediapipe_model_paths["face_landmarker"]),
        running_mode=mp_vision.RunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.5,
    )
)
hand_landmarker = mp_vision.HandLandmarker.create_from_options(
    mp_vision.HandLandmarkerOptions(
        base_options=_base_opts(mediapipe_model_paths["hand_landmarker"]),
        running_mode=mp_vision.RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.5,
    )
)
pose_landmarker = mp_vision.PoseLandmarker.create_from_options(
    mp_vision.PoseLandmarkerOptions(
        base_options=_base_opts(mediapipe_model_paths["pose_landmarker"]),
        running_mode=mp_vision.RunningMode.IMAGE,
        num_poses=1,
    )
)
print("FaceLandmarker / HandLandmarker / PoseLandmarker ready (IMAGE running mode).")


In [ ]:
# --- 3.3 PlacementAnchor payload + shared helpers ------------------------------------------------
from dataclasses import dataclass
from typing import Optional, Tuple

@dataclass
class PlacementAnchor:
    category: str
    anchor_xy_norm: Tuple[float, float]   # normalised [0,1] image coordinates
    scale_reference_px: float             # a real, measured pixel span used to convert asset mm -> px
    rotation_angle_deg: float
    occlusion_mask: Optional[np.ndarray]
    confidence: float


def _landmark_xy(landmark, w, h):
    return np.array([landmark.x * w, landmark.y * h], dtype=np.float32)


def _run_mediapipe(image_rgb):
    """Runs all three landmarkers once per image; individual get_anchor_* functions only read the
    outputs relevant to their category. mp.Image expects contiguous RGB uint8 data."""
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(image_rgb))
    return (
        face_landmarker.detect(mp_image),
        hand_landmarker.detect(mp_image),
        pose_landmarker.detect(mp_image),
    )


def _pick_hand(hand_result, side):
    """MediaPipe's Left/Right handedness label is from the SUBJECT's own perspective — flagged here
    since it's a common source of mixed-up left/right placement in a mirrored selfie-style capture."""
    for i, handedness in enumerate(hand_result.handedness):
        if handedness[0].category_name.lower() == side.lower():
            return i, float(handedness[0].score)
    if hand_result.hand_landmarks:
        return 0, float(hand_result.handedness[0][0].score)
    return None, 0.0


_NO_DETECTION = lambda category: PlacementAnchor(category, (0.5, 0.5), 0.0, 0.0, None, 0.0)

# Landmark index reference tables (documented, not magic numbers) -----------------------------------
FACE_IDX = {
    "nose_tip": 4, "chin_bottom": 152, "forehead_center": 10,
    "left_eye": 33, "right_eye": 263,
    "left_ear_approx": 132, "right_ear_approx": 361,          # nearest cheek/contour point — face
                                                                # mesh has no true earlobe landmark
    "left_eyebrow_inner": 55, "right_eyebrow_inner": 285,
    "left_nostril_approx": 129, "right_nostril_approx": 358,
}
HAND_IDX = {
    "wrist": 0, "index_mcp": 5, "middle_mcp": 9, "ring_mcp": 13, "ring_pip": 14, "pinky_mcp": 17,
}
POSE_IDX = {
    "left_shoulder": 11, "right_shoulder": 12,
    "left_ankle": 27, "right_ankle": 28,
    "left_foot_index": 31, "right_foot_index": 32,
}


In [ ]:
# --- 3.4 One function per jewellery category (per the brief's category list) --------------------

def get_anchor_earrings(image_rgb: np.ndarray, side: str = "left") -> PlacementAnchor:
    """Anchor: approximate earlobe point, taken as the nearest MediaPipe Face Landmarker
    cheek/contour landmark. FLAGGED APPROXIMATION: the 468-point face mesh does not model the
    ear/earlobe at all (it's a face-surface mesh) — production would want per-brand calibration
    or an ear-specific detector; this is a documented, workable proxy, not exact earlobe geometry."""
    h, w = image_rgb.shape[:2]
    face_result, _, _ = _run_mediapipe(image_rgb)
    if not face_result.face_landmarks:
        return _NO_DETECTION("earrings")

    lms = face_result.face_landmarks[0]
    idx = FACE_IDX["left_ear_approx"] if side == "left" else FACE_IDX["right_ear_approx"]
    anchor_px = _landmark_xy(lms[idx], w, h)

    left_eye = _landmark_xy(lms[FACE_IDX["left_eye"]], w, h)
    right_eye = _landmark_xy(lms[FACE_IDX["right_eye"]], w, h)
    scale_ref = float(np.linalg.norm(right_eye - left_eye))  # inter-eye distance: stable facial measurement

    chin = _landmark_xy(lms[FACE_IDX["chin_bottom"]], w, h)
    forehead = _landmark_xy(lms[FACE_IDX["forehead_center"]], w, h)
    rotation = float(np.degrees(np.arctan2(chin[0] - forehead[0], chin[1] - forehead[1])))

    return PlacementAnchor("earrings", (anchor_px[0] / w, anchor_px[1] / h), scale_ref, rotation, None, 0.9)


def get_anchor_necklace_pendant(image_rgb: np.ndarray) -> PlacementAnchor:
    """Anchor: interpolated between chin (face mesh) and shoulder midpoint (pose) — where a pendant
    naturally rests. Scale reference: shoulder width. The 0.35 drop fraction is a workable default;
    production would want per-piece-type presets (choker / princess / matinee length)."""
    h, w = image_rgb.shape[:2]
    face_result, _, pose_result = _run_mediapipe(image_rgb)
    if not face_result.face_landmarks or not pose_result.pose_landmarks:
        return _NO_DETECTION("necklace_pendant")

    face_lms, pose_lms = face_result.face_landmarks[0], pose_result.pose_landmarks[0]
    chin = _landmark_xy(face_lms[FACE_IDX["chin_bottom"]], w, h)
    l_sh = _landmark_xy(pose_lms[POSE_IDX["left_shoulder"]], w, h)
    r_sh = _landmark_xy(pose_lms[POSE_IDX["right_shoulder"]], w, h)
    shoulder_mid = (l_sh + r_sh) / 2.0

    anchor_px = chin + 0.35 * (shoulder_mid - chin)
    scale_ref = float(np.linalg.norm(r_sh - l_sh))
    rotation = float(np.degrees(np.arctan2(r_sh[0] - l_sh[0], r_sh[1] - l_sh[1]))) - 90.0

    return PlacementAnchor("necklace_pendant", (anchor_px[0] / w, anchor_px[1] / h), scale_ref, rotation, None, 0.85)


def get_anchor_bracelet_watch(image_rgb: np.ndarray, side: str = "left") -> PlacementAnchor:
    """Anchor: wrist point (Hand Landmarker). Scale reference: estimated wrist width, ~0.6x the
    index-to-pinky MCP knuckle span. FLAGGED APPROXIMATION: hand landmarks measure knuckle span, not
    wrist circumference directly — this calibration constant should be replaced with a measured
    wrist-width prior once real capture data is available."""
    h, w = image_rgb.shape[:2]
    _, hand_result, _ = _run_mediapipe(image_rgb)
    if not hand_result.hand_landmarks:
        return _NO_DETECTION("bracelet_watch")

    hand_idx, confidence = _pick_hand(hand_result, side)
    if hand_idx is None:
        return _NO_DETECTION("bracelet_watch")

    lms = hand_result.hand_landmarks[hand_idx]
    wrist = _landmark_xy(lms[HAND_IDX["wrist"]], w, h)
    idx_mcp = _landmark_xy(lms[HAND_IDX["index_mcp"]], w, h)
    pinky_mcp = _landmark_xy(lms[HAND_IDX["pinky_mcp"]], w, h)
    middle_mcp = _landmark_xy(lms[HAND_IDX["middle_mcp"]], w, h)

    scale_ref = float(np.linalg.norm(pinky_mcp - idx_mcp)) * 0.6
    rotation = float(np.degrees(np.arctan2(middle_mcp[0] - wrist[0], middle_mcp[1] - wrist[1])))

    return PlacementAnchor("bracelet_watch", (wrist[0] / w, wrist[1] / h), scale_ref, rotation, None, confidence)


def get_anchor_ring(image_rgb: np.ndarray, side: str = "left", finger: str = "ring") -> PlacementAnchor:
    """Anchor: midpoint of the chosen finger's MCP-PIP segment (base of the finger — where rings
    conventionally sit). Scale reference: average finger width, approximated as
    (index_MCP-to-pinky_MCP span) / 4."""
    h, w = image_rgb.shape[:2]
    _, hand_result, _ = _run_mediapipe(image_rgb)
    if not hand_result.hand_landmarks:
        return _NO_DETECTION("ring")

    hand_idx, confidence = _pick_hand(hand_result, side)
    if hand_idx is None:
        return _NO_DETECTION("ring")

    lms = hand_result.hand_landmarks[hand_idx]
    finger_joints = {"ring": (13, 14), "index": (5, 6), "middle": (9, 10), "pinky": (17, 18)}
    mcp_i, pip_i = finger_joints[finger]
    mcp, pip = _landmark_xy(lms[mcp_i], w, h), _landmark_xy(lms[pip_i], w, h)
    anchor_px = (mcp + pip) / 2.0

    idx_mcp = _landmark_xy(lms[HAND_IDX["index_mcp"]], w, h)
    pinky_mcp = _landmark_xy(lms[HAND_IDX["pinky_mcp"]], w, h)
    scale_ref = float(np.linalg.norm(pinky_mcp - idx_mcp)) / 4.0
    rotation = float(np.degrees(np.arctan2(pip[0] - mcp[0], pip[1] - mcp[1])))

    return PlacementAnchor("ring", (anchor_px[0] / w, anchor_px[1] / h), scale_ref, rotation, None, confidence)


def get_anchor_anklet(image_rgb: np.ndarray, side: str = "left") -> PlacementAnchor:
    """Anchor: ankle landmark (Pose Landmarker). Scale reference: ankle-to-foot-index distance as a
    proxy for ankle circumference (BlazePose's 33 landmarks don't include ankle width directly)."""
    h, w = image_rgb.shape[:2]
    _, _, pose_result = _run_mediapipe(image_rgb)
    if not pose_result.pose_landmarks:
        return _NO_DETECTION("anklet")

    lms = pose_result.pose_landmarks[0]
    ankle_key = "left_ankle" if side == "left" else "right_ankle"
    foot_key = "left_foot_index" if side == "left" else "right_foot_index"
    ankle = _landmark_xy(lms[POSE_IDX[ankle_key]], w, h)
    foot = _landmark_xy(lms[POSE_IDX[foot_key]], w, h)

    scale_ref = float(np.linalg.norm(foot - ankle)) * 0.5
    rotation = float(np.degrees(np.arctan2(foot[0] - ankle[0], foot[1] - ankle[1])))

    return PlacementAnchor("anklet", (ankle[0] / w, ankle[1] / h), scale_ref, rotation, None, 0.8)


def get_anchor_bangle_stack(image_rgb: np.ndarray, side: str = "left") -> PlacementAnchor:
    """Same wrist anchor as bracelet_watch — a bangle STACK's per-item vertical offsets along the
    forearm are a Stage-3 placement concern (repeat the asset N times with an offset), not a
    Stage-2 detection concern, so this deliberately reuses get_anchor_bracelet_watch rather than
    duplicating landmark logic."""
    anchor = get_anchor_bracelet_watch(image_rgb, side=side)
    anchor.category = "bangle_stack"
    return anchor


def get_anchor_maang_tikka(image_rgb: np.ndarray) -> PlacementAnchor:
    """Anchor: hairline centre-parting point. FLAGGED EXTRAPOLATION: the 468-point face mesh stops
    at the forehead/hairline boundary (landmark 10) and does not model hair/scalp at all, so the
    parting point is a controlled extrapolation — continue the eyebrow-midpoint -> forehead-centre
    vector upward — not a direct landmark read. Confidence is deliberately lower for this category."""
    h, w = image_rgb.shape[:2]
    face_result, _, _ = _run_mediapipe(image_rgb)
    if not face_result.face_landmarks:
        return _NO_DETECTION("maang_tikka")

    lms = face_result.face_landmarks[0]
    l_brow = _landmark_xy(lms[FACE_IDX["left_eyebrow_inner"]], w, h)
    r_brow = _landmark_xy(lms[FACE_IDX["right_eyebrow_inner"]], w, h)
    brow_mid = (l_brow + r_brow) / 2.0
    forehead = _landmark_xy(lms[FACE_IDX["forehead_center"]], w, h)
    direction = forehead - brow_mid
    anchor_px = forehead + 0.9 * direction

    left_eye = _landmark_xy(lms[FACE_IDX["left_eye"]], w, h)
    right_eye = _landmark_xy(lms[FACE_IDX["right_eye"]], w, h)
    scale_ref = float(np.linalg.norm(right_eye - left_eye)) * 0.5
    rotation = float(np.degrees(np.arctan2(direction[0], direction[1])))

    return PlacementAnchor("maang_tikka", (anchor_px[0] / w, anchor_px[1] / h), scale_ref, rotation, None, 0.6)


def get_anchor_nose_ring(image_rgb: np.ndarray, side: str = "left") -> PlacementAnchor:
    """Anchor: approximate nostril/ala landmark. `side` picks left or right (nose studs/rings are
    worn on either side)."""
    h, w = image_rgb.shape[:2]
    face_result, _, _ = _run_mediapipe(image_rgb)
    if not face_result.face_landmarks:
        return _NO_DETECTION("nose_ring")

    lms = face_result.face_landmarks[0]
    idx = FACE_IDX["left_nostril_approx"] if side == "left" else FACE_IDX["right_nostril_approx"]
    anchor_px = _landmark_xy(lms[idx], w, h)

    left_eye = _landmark_xy(lms[FACE_IDX["left_eye"]], w, h)
    right_eye = _landmark_xy(lms[FACE_IDX["right_eye"]], w, h)
    scale_ref = float(np.linalg.norm(right_eye - left_eye)) * 0.15

    nose_tip = _landmark_xy(lms[FACE_IDX["nose_tip"]], w, h)
    forehead = _landmark_xy(lms[FACE_IDX["forehead_center"]], w, h)
    rotation = float(np.degrees(np.arctan2(nose_tip[0] - forehead[0], nose_tip[1] - forehead[1])))

    return PlacementAnchor("nose_ring", (anchor_px[0] / w, anchor_px[1] / h), scale_ref, rotation, None, 0.75)


def get_anchor_necklace_set(image_rgb: np.ndarray) -> PlacementAnchor:
    """A 'necklace set' (necklace + matching earrings sold/worn together) returns the necklace
    anchor as primary. Stage 3 orchestrates placing the matching earring asset(s) via a separate
    get_anchor_earrings() call — per the brief's payload note about a 'per-item identifier where
    multiple pieces are placed', Stage 2 stays one-anchor-per-surface and Stage 3 composes sets."""
    anchor = get_anchor_necklace_pendant(image_rgb)
    anchor.category = "necklace_set"
    return anchor


CATEGORY_ANCHOR_FUNCTIONS = {
    "earrings": get_anchor_earrings,
    "necklace_pendant": get_anchor_necklace_pendant,
    "bracelet_watch": get_anchor_bracelet_watch,
    "ring": get_anchor_ring,
    "anklet": get_anchor_anklet,
    "bangle_stack": get_anchor_bangle_stack,
    "maang_tikka": get_anchor_maang_tikka,
    "nose_ring": get_anchor_nose_ring,
    "necklace_set": get_anchor_necklace_set,
}
print(f"{len(CATEGORY_ANCHOR_FUNCTIONS)} category anchor functions registered: "
      f"{list(CATEGORY_ANCHOR_FUNCTIONS.keys())}")


## 4. Stage 3 — Deterministic Geometric Placement

**Technology:** OpenCV only — image warping, scaling, rotation. **No model calls anywhere in this
stage.** This is the stage that makes the fidelity constraint *structurally* true rather than
statistically likely: no model ever touches the piece's pixels here.

**Output:** a "hard-pasted" composite — jewellery correctly placed and scaled, but flat (no shadow,
no lighting integration yet — that's Stage 4).

In [ ]:
# --- 4.1 place_jewellery() — brief's exact function signature, plus a mask-returning companion ---
from typing import Tuple, Optional

# Calibration lookup: category -> the real-world mm value that `scale_reference_px` corresponds to.
# FLAGGED APPROXIMATION: these are population-average placeholders (documented, not silently
# assumed), since we don't have a measured-norms lookup table. Production should replace this with
# calibrated values, ideally per demographic/region given SkyAI's multi-tenant, multi-market scope.
CATEGORY_REFERENCE_MM = {
    "earrings": 63.0,           # average inter-pupillary distance
    "necklace_pendant": 360.0,  # average shoulder width
    "necklace_set": 360.0,
    "bracelet_watch": 45.0,     # average wrist width
    "bangle_stack": 45.0,
    "ring": 18.0,               # average finger width
    "anklet": 55.0,
    "maang_tikka": 31.5,
    "nose_ring": 9.5,
}


def _warp_asset(asset_rgba: np.ndarray, scale_ref: float, rotation: float,
                 physical_dimensions: Tuple[float, float], category: str) -> np.ndarray:
    """Shared scale+rotate transform used by both place_jewellery() and place_jewellery_and_mask()."""
    reference_mm = CATEGORY_REFERENCE_MM.get(category, 50.0)
    px_per_mm = scale_ref / reference_mm

    target_w = max(1, int(round(physical_dimensions[0] * px_per_mm)))
    target_h = max(1, int(round(physical_dimensions[1] * px_per_mm)))
    resized = cv2.resize(asset_rgba, (target_w, target_h), interpolation=cv2.INTER_AREA)

    center = (target_w / 2.0, target_h / 2.0)
    rot_mat = cv2.getRotationMatrix2D(center, -rotation, 1.0)
    cos, sin = abs(rot_mat[0, 0]), abs(rot_mat[0, 1])
    new_w = int(target_h * sin + target_w * cos)
    new_h = int(target_h * cos + target_w * sin)
    rot_mat[0, 2] += (new_w / 2.0) - center[0]
    rot_mat[1, 2] += (new_h / 2.0) - center[1]

    return cv2.warpAffine(resized, rot_mat, (new_w, new_h), flags=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))


def _alpha_blend_onto(portrait_rgb: np.ndarray, transformed_rgba: np.ndarray,
                       anchor_xy_norm: Tuple[float, float], category: str):
    """Places `transformed_rgba` (already scaled+rotated) onto `portrait_rgb` centred at the
    anchor point, alpha-blending at the edges. Returns (composite, full-frame placement mask)."""
    ph, pw = portrait_rgb.shape[:2]
    new_h, new_w = transformed_rgba.shape[:2]
    anchor_px = np.array([anchor_xy_norm[0] * pw, anchor_xy_norm[1] * ph])
    x0, y0 = (anchor_px - np.array([new_w / 2.0, new_h / 2.0])).astype(int)
    x1, y1 = x0 + new_w, y0 + new_h

    src_x0, src_y0 = max(0, -x0), max(0, -y0)
    dst_x0, dst_y0 = max(0, x0), max(0, y0)
    dst_x1 = min(pw, x1); dst_y1 = min(ph, y1)
    src_x1 = src_x0 + (dst_x1 - dst_x0)
    src_y1 = src_y0 + (dst_y1 - dst_y0)

    if dst_x1 <= dst_x0 or dst_y1 <= dst_y0:
        raise ValueError(f"Placement for '{category}' falls entirely outside the portrait frame — "
                          f"check the anchor/scale inputs.")

    asset_crop = transformed_rgba[src_y0:src_y1, src_x0:src_x1]
    alpha = asset_crop[:, :, 3:4].astype(np.float32) / 255.0
    fg = asset_crop[:, :, :3].astype(np.float32)

    composite = portrait_rgb.copy()
    bg = composite[dst_y0:dst_y1, dst_x0:dst_x1].astype(np.float32)
    composite[dst_y0:dst_y1, dst_x0:dst_x1] = (fg * alpha + bg * (1 - alpha)).astype(np.uint8)

    placement_mask = np.zeros((ph, pw), dtype=np.float32)
    placement_mask[dst_y0:dst_y1, dst_x0:dst_x1] = alpha[:, :, 0]
    return composite, placement_mask


def place_jewellery(asset_rgba: np.ndarray, anchor: Tuple[float, float], scale_ref: float,
                     rotation: float, physical_dimensions: Tuple[float, float],
                     portrait_rgb: np.ndarray, category: str = "generic") -> np.ndarray:
    """
    Stage 3 — matches the brief's exact signature:
    `place_jewellery(asset_rgba, anchor, scale_ref, rotation, physical_dimensions) -> composite_image`
    (`anchor` here is the normalised (x, y) point from a Stage-2 PlacementAnchor; `portrait_rgb` and
    `category` are added since compositing needs a target image and the mm->px lookup needs a
    category — the brief's Stage-2 payload already carries category alongside these fields anyway).

    NO model call anywhere in this function — every transform is closed-form affine math. This is
    what makes the piece's shape/gem-cut/proportion fidelity a *structural* guarantee, per the
    brief's core design rationale.
    """
    transformed = _warp_asset(asset_rgba, scale_ref, rotation, physical_dimensions, category)
    composite, _ = _alpha_blend_onto(portrait_rgb, transformed, anchor, category)
    return composite


def place_jewellery_and_mask(asset_rgba: np.ndarray, anchor: 'PlacementAnchor',
                              physical_dimensions: Tuple[float, float],
                              portrait_rgb: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Convenience companion to place_jewellery(): also returns the full-frame placement alpha
    mask, which Stage 4 needs to build its boundary/shadow mask. Takes a full PlacementAnchor
    object rather than its unpacked fields, since Stage 4 also wants `anchor.confidence` for its
    own pre-flight check."""
    if anchor.confidence <= 0.0:
        raise ValueError(
            f"Cannot place '{anchor.category}': Stage 2 returned zero confidence (no landmarks "
            f"detected). Retry the capture rather than rendering a low-confidence/undefined fit."
        )
    transformed = _warp_asset(asset_rgba, anchor.scale_reference_px, anchor.rotation_angle_deg,
                               physical_dimensions, anchor.category)
    return _alpha_blend_onto(portrait_rgb, transformed, anchor.anchor_xy_norm, anchor.category)


## 5. Stage 4 — Bounded Generative Harmonisation (the only generative stage)

Goal: add contact shadow, ambient occlusion, and lighting/colour match around the piece — **without
ever regenerating the piece itself**.

- **Primary candidate:** `black-forest-labs/FLUX.1-Fill-dev` — a dedicated inpainting checkpoint with
  a real trained-in mask channel, giving a stronger architectural guarantee that pixels outside the
  mask stay untouched.
- **A/B candidate:** `Qwen/Qwen-Image-Edit-2511` — regarded as very strong at preserving everything
  outside the edited region, tested head-to-head below using the same fine-tuning recipe.
- **Fine-tuning:** a custom edit LoRA trained via `ostris/ai-toolkit`, so the model acts strictly as
  a compositor/lighting-fixer, not a general-purpose generator.

> **License note:** `FLUX.1-Fill-dev` ships under the FLUX.1 [dev] Non-Commercial License (see the
> model card at https://huggingface.co/black-forest-labs/FLUX.1-Fill-dev). Fine for this prototyping
> notebook; **not** cleared for revenue-generating production use as-is — licensing is deliberately
> out of scope for this brief, flagged here so it isn't missed downstream.

In [ ]:
# --- 5.1 Load FLUX.1 Fill [dev] -------------------------------------------------------------------
from diffusers import FluxFillPipeline

FLUX_FILL_REPO = "black-forest-labs/FLUX.1-Fill-dev"
flux_fill_pipe = None

def load_flux_fill():
    global flux_fill_pipe
    if flux_fill_pipe is not None:
        return flux_fill_pipe
    flux_fill_pipe = FluxFillPipeline.from_pretrained(FLUX_FILL_REPO, torch_dtype=torch.bfloat16)
    if device == "cuda":
        if USE_CPU_OFFLOAD_FALLBACK:
            print("[INFO] VRAM-constrained: using enable_model_cpu_offload() instead of pipe.to('cuda'). "
                  "For a true fp8-weights fallback, quantize the transformer with `optimum-quanto` "
                  "(qfloat8) before wrapping it in the pipeline — left as a documented extension "
                  "point rather than hard-baked here, since it needs an extra dependency and a "
                  "slightly different loading path.")
            flux_fill_pipe.enable_model_cpu_offload()
        else:
            flux_fill_pipe.to(device)
    return flux_fill_pipe

print("FLUX.1 Fill [dev] loader ready (lazy — call load_flux_fill() when Stage 4 actually runs).")


In [ ]:
# --- 5.2 Boundary/shadow mask: dilate the Stage-1 alpha outward, subtract the original -----------
def build_boundary_shadow_mask(jewellery_alpha_mask: np.ndarray, dilate_margin_px: int = 40) -> np.ndarray:
    """The ring-shaped region Stage 4 is allowed to touch: dilate the placed jewellery's alpha mask
    outward by `dilate_margin_px`, then subtract the original mask — giving a boundary/contact-
    shadow region that EXPLICITLY excludes the jewellery's own pixels."""
    binary = (jewellery_alpha_mask > 0.5).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_margin_px * 2 + 1,) * 2)
    dilated = cv2.dilate(binary, kernel)
    return cv2.subtract(dilated, binary)  # uint8, 0/255 — passed as mask_image to FluxFillPipeline


In [ ]:
# --- 5.3 harmonise() -------------------------------------------------------------------------------
from PIL import Image

DEFAULT_HARMONISE_PROMPT = "harmonize lighting and add realistic contact shadows to the jewelry."

def harmonise(composite_image: np.ndarray, jewellery_mask: np.ndarray, lora_path: Optional[str] = None,
              dilate_margin_px: int = 40, guidance_scale: float = 30.0,
              num_inference_steps: int = 30, seed: Optional[int] = None) -> np.ndarray:
    """
    Stage 4 — the ONLY generative stage. Adds contact shadow / ambient occlusion / lighting-colour
    match via FLUX.1 Fill's masked inpainting. The jewellery region is excluded from the mask, so
    FLUX Fill's trained-in mask channel is the structural guarantee those pixels aren't touched by
    the model — matching the brief's "mask gives fidelity, LoRA gives quality; neither alone is
    enough" recommendation.
    """
    pipe = load_flux_fill()

    # <<< LoRA load point — call this once a fine-tuned checkpoint exists (Section 5.4/5.5) >>>
    if lora_path is not None:
        pipe.load_lora_weights(lora_path)
        print(f"[INFO] Loaded harmonisation LoRA from {lora_path}")

    boundary_mask = build_boundary_shadow_mask(jewellery_mask, dilate_margin_px)

    pil_image = Image.fromarray(composite_image)
    pil_mask = Image.fromarray(boundary_mask)

    h, w = composite_image.shape[:2]
    round16 = lambda v: max(16, int(round(v / 16.0)) * 16)  # FLUX prefers multiple-of-16 dimensions
    gen_w, gen_h = round16(w), round16(h)
    if (gen_w, gen_h) != (w, h):
        pil_image = pil_image.resize((gen_w, gen_h))
        pil_mask = pil_mask.resize((gen_w, gen_h))

    generator = torch.Generator(device="cpu").manual_seed(seed) if seed is not None else None

    result = pipe(
        prompt=DEFAULT_HARMONISE_PROMPT,
        image=pil_image,
        mask_image=pil_mask,
        height=gen_h, width=gen_w,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        max_sequence_length=512,
        generator=generator,
    ).images[0]

    result_np = np.array(result.resize((w, h)))

    # Belt-and-braces: even though FLUX Fill's mask channel should already guarantee this, hard-
    # composite the ORIGINAL jewellery pixels back in using the exact Stage-1/3 alpha, so downstream
    # fidelity checks (SSIM/LPIPS/ΔE) validate something pixel-identical inside the piece BY
    # CONSTRUCTION — not just "close" per the model's own masking behaviour.
    alpha3 = np.repeat((jewellery_mask > 0.5).astype(np.float32)[:, :, None], 3, axis=2)
    return (composite_image.astype(np.float32) * alpha3 +
            result_np.astype(np.float32) * (1 - alpha3)).astype(np.uint8)

print("harmonise() ready.")


### 5.4 Fine-tuning: edit LoRA via `ostris/ai-toolkit`

**Dataset:** 50–150 paired before/after examples.
- **Input (reference):** the Stage-3 hard-pasted composite (flat, no shadow).
- **Target (output):** the same image, professionally retouched with realistic contact shadow,
  ambient occlusion, and lighting reflection — jewellery pixels identical to the input.

**Captioning:** a single, consistent instruction phrase across the dataset (not descriptive tags):
`"harmonize lighting and add realistic contact shadows to the jewelry."`

**Dataset layout** (matched by filename stem):
```
datasets/reference/000.png   datasets/target/000.png  datasets/target/000.txt
datasets/reference/001.png   datasets/target/001.png  datasets/target/001.txt
...
```

**Compute:** a 4B-class model trains in roughly 45–60 minutes on a single 24GB-class GPU (RTX 4090,
or an A100/L4 on Colab Pro) for this dataset size.

**Checkpointing:** every 250 steps, to Drive — survives a Colab disconnect (re-running the launch
cell after reconnecting resumes from the latest ai-toolkit checkpoint automatically).

**Visual peak note:** inspect sample images rather than trusting final loss alone — peak quality for
this kind of LoRA is typically around step 750–1500, not the final step.

In [ ]:
# --- 5.5 LoRA dataset prep, config generation, and training launch --------------------------------
import subprocess, sys, yaml

def ensure_ai_toolkit():
    if not AI_TOOLKIT_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/ostris/ai-toolkit.git", str(AI_TOOLKIT_DIR)], check=True)
        subprocess.run(["git", "-C", str(AI_TOOLKIT_DIR), "submodule", "update", "--init", "--recursive"], check=True)
        req_file = AI_TOOLKIT_DIR / "requirements.txt"
        if req_file.exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "--break-system-packages", "-q",
                             "-r", str(req_file)], check=False)
    else:
        print(f"[INFO] ai-toolkit already present at {AI_TOOLKIT_DIR}")


HARMONISE_CAPTION = DEFAULT_HARMONISE_PROMPT

def write_caption_files(target_dir=None, caption=HARMONISE_CAPTION):
    """A SINGLE consistent instruction phrase across the whole dataset (not descriptive per-image
    tags) — per the brief, this teaches the model to behave like a fixed-function compositor rather
    than a general text-to-image model that happens to be LoRA-tuned."""
    target_dir = Path(target_dir or DATASET_TARGET_DIR)
    n = 0
    for img_path in sorted(target_dir.glob("*.png")):
        cap_path = img_path.with_suffix(".txt")
        if not cap_path.exists():
            cap_path.write_text(caption)
            n += 1
    print(f"Wrote {n} caption file(s) to {target_dir}")


def build_ai_toolkit_config(run_name: str = "skyai_harmonisation_lora",
                             base_model: str = FLUX_FILL_REPO,
                             network_rank: int = 16, learning_rate: float = 1e-4,
                             steps: int = 1200, checkpoint_every: int = 250,
                             resolution=(512, 768)) -> dict:
    """
    Generates the ai-toolkit YAML job config. `control_path` points at the hard-pasted reference
    folder, `folder_path` at the retouched target folder — this pairing is what turns a style LoRA
    into an EDIT LoRA in ai-toolkit.

    FLAGGED, NOT SILENTLY ASSUMED: ai-toolkit's set of supported `model.arch` strings (flux_kontext,
    qwen_image, etc.) changes fairly often as new edit-style base models land, and there was no
    confirmed dedicated `arch` value for FLUX.1-Fill specifically in ai-toolkit's public examples at
    the time this notebook was written. "flux_fill" below follows the naming pattern of neighbouring
    archs (flux_kontext, flux_dev) as a best-effort default — VERIFY against
    `ai-toolkit/config/examples/` (the flux_kontext example is the closest documented analogue for
    an edit/control-conditioned Flux variant) before trusting a real training run.
    """
    return {
        "job": "extension",
        "config": {
            "name": run_name,
            "process": [{
                "type": "sd_trainer",
                "training_folder": str(LORA_OUTPUT_DIR),
                "device": "cuda:0",
                "datasets": [{
                    "folder_path": str(DATASET_TARGET_DIR),      # retouched targets (desired output)
                    "control_path": str(DATASET_REFERENCE_DIR),  # hard-pasted references (model input)
                    "caption_ext": "txt",
                    "caption_dropout_rate": 0.0,   # keep the single instruction caption every step —
                                                     # it is not a style trigger word to drop out
                    "shuffle_tokens": False,
                    "cache_latents_to_disk": True,
                    "resolution": list(resolution),
                }],
                "train": {
                    "batch_size": 1,
                    "steps": steps,
                    "gradient_accumulation_steps": 1,
                    "train_unet": True,
                    "train_text_encoder": False,
                    "gradient_checkpointing": True,
                    "noise_scheduler": "flowmatch",
                    "optimizer": "adamw8bit",
                    "lr": learning_rate,
                    "dtype": "bf16",
                },
                "model": {
                    "name_or_path": base_model,
                    "arch": "flux_fill",  # <-- see docstring: VERIFY before a real run
                    "quantize": True,      # quantize the frozen base weights to fit training in less VRAM
                },
                "network": {"type": "lora", "linear": network_rank, "linear_alpha": network_rank},
                "save": {
                    "dtype": "float16",
                    "save_every": checkpoint_every,          # Drive checkpoint cadence, per the brief
                    "max_step_saves_to_keep": 20,
                },
                "sample": {"sample_every": checkpoint_every, "prompts": [HARMONISE_CAPTION]},
            }],
        },
    }


def launch_lora_training(config: dict):
    """Writes the YAML config and invokes ai-toolkit's run.py via subprocess. ai-toolkit resumes
    from the latest checkpoint under training_folder/run_name automatically on the next invocation
    if one exists — this is the mechanism this notebook relies on for surviving a Colab disconnect;
    re-run this cell after reconnecting rather than deleting the Drive output folder."""
    ensure_ai_toolkit()
    run_name = config["config"]["name"]
    config_path = AI_TOOLKIT_DIR / "config" / f"{run_name}.yaml"
    config_path.parent.mkdir(parents=True, exist_ok=True)
    with open(config_path, "w") as f:
        yaml.safe_dump(config, f, sort_keys=False)
    print(f"Wrote training config -> {config_path}")

    cmd = [sys.executable, "run.py", str(config_path)]
    print(f"Launching: {' '.join(cmd)}  (cwd={AI_TOOLKIT_DIR})")
    return subprocess.Popen(cmd, cwd=str(AI_TOOLKIT_DIR))


print("Compute estimate: ~45-60 min on a single 24GB-class GPU for this dataset size (per the brief).")
print("Inspect SAMPLE IMAGES, not just final loss — visual peak is typically step 750-1500.")

# Uncomment to actually train once DATASET_REFERENCE_DIR / DATASET_TARGET_DIR hold real paired data
# (Section 7 shows how the placeholder demo pair maps onto this folder structure):
#
# write_caption_files()
# lora_config = build_ai_toolkit_config()
# training_process = launch_lora_training(lora_config)


### 5.6 A/B candidate: Qwen-Image-Edit-2511

Per the brief, this must be tested head-to-head against FLUX.1 Fill using the **same** fine-tuning
recipe — same dataset, same single-instruction caption — letting measured SSIM/LPIPS/ΔE results pick
the production backbone rather than assuming FLUX wins.

**Key structural difference to work around:** `QwenImageEditPlusPipeline` (unlike `FluxFillPipeline`)
has **no native `mask_image` channel** — it edits the whole image from an instruction prompt. So the
brief's "mask-constrained sampling" is implemented here as a **post-hoc masked blend**: run the edit
on the full composite, then hard-composite the untouched jewellery pixels back in using the exact
Stage-1 alpha. For FLUX Fill that hard re-composite was a *backstop* on top of a real mask channel;
for Qwen it is the **only** fidelity guarantee available — which is exactly the trade-off the brief
flags between FLUX Fill's stronger architectural guarantee and Qwen's stronger general
preservation-outside-the-edit behaviour.

In [ ]:
# --- 5.7 Qwen-Image-Edit-2511 — minimal parallel implementation for A/B testing --------------------
from diffusers import QwenImageEditPlusPipeline

QWEN_EDIT_REPO = "Qwen/Qwen-Image-Edit-2511"
qwen_edit_pipe = None

def load_qwen_edit():
    global qwen_edit_pipe
    if qwen_edit_pipe is not None:
        return qwen_edit_pipe
    qwen_edit_pipe = QwenImageEditPlusPipeline.from_pretrained(QWEN_EDIT_REPO, torch_dtype=torch.bfloat16)
    if device == "cuda":
        qwen_edit_pipe.enable_model_cpu_offload() if USE_CPU_OFFLOAD_FALLBACK else qwen_edit_pipe.to(device)
    return qwen_edit_pipe


def harmonise_qwen(composite_image: np.ndarray, jewellery_mask: np.ndarray, lora_path: Optional[str] = None,
                    num_inference_steps: int = 30, seed: Optional[int] = None) -> np.ndarray:
    pipe = load_qwen_edit()
    if lora_path is not None:
        pipe.load_lora_weights(lora_path)

    pil_image = Image.fromarray(composite_image)
    generator = torch.Generator(device="cpu").manual_seed(seed) if seed is not None else None

    result = pipe(
        image=pil_image,
        prompt=DEFAULT_HARMONISE_PROMPT,
        num_inference_steps=num_inference_steps,
        generator=generator,
    ).images[0]
    result_np = np.array(result.resize((composite_image.shape[1], composite_image.shape[0])))

    # Post-hoc hard mask — see the markdown note above on why this carries the FULL fidelity
    # guarantee for this backbone (no native mask channel to also rely on), not just a backstop.
    alpha3 = np.repeat((jewellery_mask > 0.5).astype(np.float32)[:, :, None], 3, axis=2)
    return (composite_image.astype(np.float32) * alpha3 +
            result_np.astype(np.float32) * (1 - alpha3)).astype(np.uint8)


def build_ai_toolkit_config_qwen(run_name: str = "skyai_harmonisation_lora_qwen", **kwargs) -> dict:
    """Same recipe as build_ai_toolkit_config(), swapped to the Qwen base. Same VERIFY-before-trusting
    caveat applies to the `arch` key — check ai-toolkit's qwen_image-family examples."""
    cfg = build_ai_toolkit_config(run_name=run_name, base_model=QWEN_EDIT_REPO, **kwargs)
    cfg["config"]["process"][0]["model"]["arch"] = "qwen_image_edit"  # VERIFY against ai-toolkit examples
    return cfg

print("harmonise_qwen() ready — same call shape as harmonise(), for a direct A/B comparison in Section 6.")


## 6. Validation / Acceptance Targets

Measured, not eyeballed, per the brief:
- **Visual similarity** of the placed piece vs. the source-extracted asset: **SSIM/LPIPS ≥ 0.95**
- **Colour drift** on the piece: **ΔE ≤ 2**
- **Production acceptance rate** (catalogue renders approved with no manual retouch): **≥ 90%**

In [ ]:
# --- 6.1 evaluate_fidelity() and color_delta_e() ---------------------------------------------------
from skimage.metrics import structural_similarity as ssim_metric
from skimage.color import rgb2lab, deltaE_cie76
import lpips as lpips_lib

_lpips_model = None

def _get_lpips():
    global _lpips_model
    if _lpips_model is None:
        _lpips_model = lpips_lib.LPIPS(net='alex').to(device)
    return _lpips_model


def _crop_to_mask_bbox(image: np.ndarray, mask: np.ndarray, pad: int = 4):
    ys, xs = np.where(mask > 0.5)
    if len(ys) == 0:
        return image, mask
    y0, y1 = max(0, ys.min() - pad), min(image.shape[0], ys.max() + pad + 1)
    x0, x1 = max(0, xs.min() - pad), min(image.shape[1], xs.max() + pad + 1)
    return image[y0:y1, x0:x1], mask[y0:y1, x0:x1]


def evaluate_fidelity(placed_asset: np.ndarray, source_asset: np.ndarray,
                       mask: Optional[np.ndarray] = None) -> dict:
    """SSIM and LPIPS between the jewellery region of the final render and the original Stage-1
    segmented asset. Both inputs are cropped to the mask bounding box (if given) and resized to a
    common size so the metrics compare like-for-like regardless of placement scale."""
    if mask is not None:
        placed_asset, _ = _crop_to_mask_bbox(placed_asset, mask)
        source_asset, _ = _crop_to_mask_bbox(source_asset, mask)

    target_size = (256, 256)
    a = cv2.resize(placed_asset, target_size)
    b = cv2.resize(source_asset, target_size)

    ssim_score = float(ssim_metric(a, b, channel_axis=2, data_range=255))

    lp = _get_lpips()
    to_tensor = lambda img: (torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0) / 127.5 - 1.0).to(device)
    with torch.no_grad():
        lpips_score = float(lp(to_tensor(a), to_tensor(b)).item())

    return {"ssim": ssim_score, "lpips": lpips_score}


def color_delta_e(placed_asset: np.ndarray, source_asset: np.ndarray, mask: Optional[np.ndarray] = None) -> float:
    """Mean CIE76 ΔE between the placed jewellery region and the source asset, in CIELAB — via
    scikit-image rather than the (effectively unmaintained) `colormath` package. FLAGGED DEVIATION
    from a literal reading of "e.g. via colormath": functionally equivalent CIE76 ΔE math, actively
    maintained dependency, per the build brief's own colormath-OR-skimage allowance."""
    if mask is not None:
        placed_asset, _ = _crop_to_mask_bbox(placed_asset, mask)
        source_asset, _ = _crop_to_mask_bbox(source_asset, mask)

    target_size = (256, 256)
    a_lab = rgb2lab(cv2.resize(placed_asset, target_size) / 255.0)
    b_lab = rgb2lab(cv2.resize(source_asset, target_size) / 255.0)
    return float(np.mean(deltaE_cie76(a_lab, b_lab)))


In [ ]:
# --- 6.2 Pass/fail report against the brief's targets ----------------------------------------------
ACCEPTANCE_TARGETS = {"ssim_min": 0.95, "lpips_max": 0.05, "delta_e_max": 2.0}
# NOTE ON READING THE BRIEF'S TARGETS: the brief states "SSIM/LPIPS >= 0.95" for both metrics
# identically. SSIM's "good" direction is >= (1.0 = identical) — taken literally below. LPIPS is a
# *distance* (0.0 = identical; genuinely similar image pairs essentially never score >= 0.95), so a
# literal ">=0.95" reading would make the LPIPS check unpassable by construction. Interpreted here as
# a tight LPIPS ceiling (<= 0.05) instead — flagged explicitly rather than silently picking a
# direction, per the build brief's own instruction not to substitute without flagging it.

def print_fidelity_report(metrics: dict, delta_e: float, targets: dict = ACCEPTANCE_TARGETS) -> bool:
    status = lambda ok: "PASS" if ok else "FAIL"
    ssim_ok = metrics["ssim"] >= targets["ssim_min"]
    lpips_ok = metrics["lpips"] <= targets["lpips_max"]
    de_ok = delta_e <= targets["delta_e_max"]

    print("Fidelity report")
    print("-" * 44)
    print(f"SSIM      : {metrics['ssim']:.4f}  (target >= {targets['ssim_min']})   [{status(ssim_ok)}]")
    print(f"LPIPS     : {metrics['lpips']:.4f}  (target <= {targets['lpips_max']})   [{status(lpips_ok)}]")
    print(f"Color dE  : {delta_e:.4f}  (target <= {targets['delta_e_max']})   [{status(de_ok)}]")
    print("-" * 44)
    overall = ssim_ok and lpips_ok and de_ok
    print(f"Overall   : [{status(overall)}]")
    return overall


def estimate_acceptance_rate(pass_fail_flags: list) -> float:
    """'Production acceptance rate >= 90%' is a business-process KPI (% of catalogue renders a
    human reviewer approves with no manual retouch) — it needs a labelled review batch, not just
    per-image SSIM/LPIPS/ΔE, so this notebook can't compute it from a single demo render. This stub
    is the aggregation point once real per-render pass/fail (or human-review) flags start flowing in."""
    if not pass_fail_flags:
        return 0.0
    return 100.0 * sum(1 for f in pass_fail_flags if f) / len(pass_fail_flags)


## 7. Demo & Data

Real catalogue/portrait data isn't available in this environment, so this section sources or
generates a small placeholder set so the notebook runs end-to-end:

- **Portrait:** sourced from Google's own public MediaPipe sample-asset bucket (a real photograph —
  needed because Stage 2's neural landmarkers expect photographic features, not synthetic art; a
  hand-drawn cartoon face will not reliably detect). Falls back to a synthetic placeholder only if
  no internet access is available, with a clear warning that Stage 2 will then likely report zero
  confidence (by design — see `_NO_DETECTION` in Section 3.3).
- **Jewellery piece:** procedurally generated (a simple stud/drop earring on a plain background) —
  clearly commented as a stand-in for a real macro product photo. Segmentation quality here is not
  representative of real filigree/chain performance; that's exactly why Stage 1 calls for
  jewellery-specific fine-tuning in production.

**Where real tenant data would substitute:** replace `portrait_rgb` and `jewellery_rgb` below with a
brand's actual catalogue photo and a shopper's actual capture — nothing else in Sections 2-6 changes.

In [ ]:
# --- 7.1 Portrait placeholder: source a real sample photo, with a synthetic fallback --------------
import urllib.request

PORTRAIT_CANDIDATE_URLS = [
    "https://storage.googleapis.com/mediapipe-assets/portrait.jpg",  # Google's own public FaceLandmarker demo asset
]

def _download_first_working(urls, dest_path):
    for url in urls:
        try:
            urllib.request.urlretrieve(url, dest_path)
            print(f"[OK] Downloaded demo portrait from {url}")
            return True
        except Exception as e:
            print(f"[WARN] Could not fetch {url}: {e}")
    return False


def _synthetic_portrait_placeholder(w=768, h=1024) -> np.ndarray:
    """Last-resort fallback if there's no internet access at all. This is a crude geometric
    placeholder, NOT photographically realistic — MediaPipe's neural landmarkers are very unlikely
    to detect a face/hand/pose on it, so Stage 2 will correctly report zero confidence for most
    categories here rather than fabricate a fake anchor. It exists purely so the notebook's
    control flow can still be exercised offline; it is not a substitute for the real sourced image."""
    print("[WARN] Using a SYNTHETIC portrait placeholder — Stage 2 landmark detection is expected "
          "to fail on this (by design; see docstring). Restore internet access for a real demo run.")
    img = np.full((h, w, 3), 235, dtype=np.uint8)
    cv2.ellipse(img, (w // 2, h // 3), (w // 6, int(w // 6 * 1.3)), 0, 0, 360, (200, 170, 150), -1)
    return img


portrait_path = DEMO_ASSETS_DIR / "portrait.jpg"
if not portrait_path.exists():
    ok = _download_first_working(PORTRAIT_CANDIDATE_URLS, portrait_path)
    if not ok:
        Image.fromarray(_synthetic_portrait_placeholder()).save(portrait_path)

portrait_bgr = cv2.imread(str(portrait_path))
portrait_rgb = cv2.cvtColor(portrait_bgr, cv2.COLOR_BGR2RGB)
print(f"Portrait loaded: {portrait_rgb.shape}")


In [ ]:
# --- 7.2 Jewellery placeholder: a procedurally drawn stud/drop earring ----------------------------
# COMMENT PER THE BRIEF: this is a synthetic stand-in for a real macro product photo — in production
# this would be a brand's actual catalogue shot. Swap `jewellery_rgb` / `jewellery_box_prompt` below
# for a real upload and nothing downstream changes.

def make_placeholder_earring(size=400) -> np.ndarray:
    img = np.full((size, size, 3), 40, dtype=np.uint8)  # dark plain background -> easy SAM2 contrast
    center = (size // 2, size // 3)

    # Metal band / post
    cv2.circle(img, center, size // 10, (212, 175, 55), -1, lineType=cv2.LINE_AA)   # gold post
    cv2.circle(img, center, size // 10, (255, 223, 120), 2, lineType=cv2.LINE_AA)   # highlight rim

    # Drop chain
    drop_end = (center[0], center[1] + size // 3)
    cv2.line(img, center, drop_end, (212, 175, 55), 4, lineType=cv2.LINE_AA)

    # Faceted "gem" at the drop end — a few overlapping triangles to fake facets
    gem_r = size // 12
    pts = []
    for k in range(6):
        ang = np.pi / 3 * k
        pts.append((int(drop_end[0] + gem_r * np.cos(ang)), int(drop_end[1] + gem_r * np.sin(ang))))
    cv2.fillConvexPoly(img, np.array(pts), (180, 230, 255))
    for k in range(6):
        cv2.line(img, drop_end, pts[k], (230, 250, 255), 1, lineType=cv2.LINE_AA)

    return img


jewellery_rgb = make_placeholder_earring()
jewellery_path = DEMO_ASSETS_DIR / "earring.png"
Image.fromarray(jewellery_rgb).save(jewellery_path)

# A box prompt covering the whole drawn piece — stands in for a click/box a real capture UI would send.
jewellery_box_prompt = {"box": [jewellery_rgb.shape[1] * 0.30, jewellery_rgb.shape[0] * 0.15,
                                 jewellery_rgb.shape[1] * 0.70, jewellery_rgb.shape[0] * 0.75]}

print(f"Placeholder jewellery asset saved to {jewellery_path}, shape={jewellery_rgb.shape}")


### 7.3 End-to-end run

Runs the placeholder portrait + earring through all four stages, displays every intermediate output
side by side, and prints the fidelity report against the brief's acceptance targets.

In [ ]:
# --- 7.4 Full pipeline, end to end ------------------------------------------------------------------
import matplotlib.pyplot as plt

DEMO_CATEGORY = "earrings"           # a plain headshot reliably gives Stage 2 a face to work with
DEMO_PHYSICAL_DIMENSIONS_MM = (10.0, 28.0)  # (width, height) of the placeholder piece — a real
                                             # catalogue asset would carry its own measured dims

print("=" * 70)
print(f"SkyAI Try-On — end-to-end run  |  category = '{DEMO_CATEGORY}'")
print("=" * 70)

# Stage 1 — segmentation ------------------------------------------------------------------------
print("\n[Stage 1] Segmenting jewellery asset...")
asset_rgba, asset_alpha, sam_score = segment_jewellery_refined(jewellery_rgb, jewellery_box_prompt)
print(f"  SAM2 mask score: {sam_score:.3f}")

# Stage 2 — anchor & landmark detection ----------------------------------------------------------
print("\n[Stage 2] Detecting placement anchor...")
anchor = CATEGORY_ANCHOR_FUNCTIONS[DEMO_CATEGORY](portrait_rgb)
print(f"  {anchor}")

anchor_overlay = portrait_rgb.copy()
if anchor.confidence > 0:
    h, w = portrait_rgb.shape[:2]
    pt = (int(anchor.anchor_xy_norm[0] * w), int(anchor.anchor_xy_norm[1] * h))
    cv2.drawMarker(anchor_overlay, pt, (255, 0, 0), markerType=cv2.MARKER_CROSS, markerSize=30, thickness=3)
    cv2.circle(anchor_overlay, pt, int(anchor.scale_reference_px * 0.1), (255, 0, 0), 2)

# Stage 3 — deterministic placement --------------------------------------------------------------
final_composite = None
hard_paste, placed_mask = None, None
if anchor.confidence > 0:
    print("\n[Stage 3] Placing jewellery (pure OpenCV, no model)...")
    hard_paste, placed_mask = place_jewellery_and_mask(asset_rgba, anchor, DEMO_PHYSICAL_DIMENSIONS_MM, portrait_rgb)
    print(f"  Hard-paste composite ready: {hard_paste.shape}")
else:
    print("\n[Stage 3] SKIPPED — Stage 2 returned zero confidence (no landmarks detected on the "
          "portrait). This is the correct behaviour (see place_jewellery_and_mask's pre-flight "
          "check) — check that a real, internet-sourced portrait loaded in Section 7.1.")

# Stage 4 — bounded generative harmonisation -------------------------------------------------------
if hard_paste is not None:
    if device == "cuda":
        print("\n[Stage 4] Harmonising with FLUX.1 Fill (this downloads a ~12B-param model on first "
              "call and can take a few minutes)...")
        try:
            final_composite = harmonise(hard_paste, placed_mask)
            print("  Harmonisation complete.")
        except Exception as e:
            print(f"[WARN] Stage 4 harmonisation failed ({e}); falling back to the Stage-3 hard-paste "
                  f"as the 'final' image so the rest of this cell can still run.")
            final_composite = hard_paste
    else:
        print("\n[Stage 4] SKIPPED — no GPU attached to this runtime, and FLUX.1 Fill on CPU is not "
              "practically usable. Showing the Stage-3 hard-paste as 'final' instead. In Colab: "
              "Runtime > Change runtime type > select a GPU, then re-run.")
        final_composite = hard_paste

# --- Display everything side by side ---------------------------------------------------------------
if hard_paste is not None:
    fig, axes = plt.subplots(1, 5, figsize=(24, 6))
    axes[0].imshow(jewellery_rgb); axes[0].set_title("Jewellery source")
    axes[1].imshow(asset_alpha, cmap="gray"); axes[1].set_title(f"Stage 1: alpha mask\n(SAM2 score {sam_score:.2f})")
    axes[2].imshow(anchor_overlay); axes[2].set_title(f"Stage 2: anchor\n(confidence {anchor.confidence:.2f})")
    axes[3].imshow(hard_paste); axes[3].set_title("Stage 3: hard-paste\n(deterministic, no model)")
    axes[4].imshow(final_composite); axes[4].set_title("Stage 4: harmonised final")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()

    out_path = RUN_OUTPUTS_DIR / "demo_end_to_end.png"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"\nSaved side-by-side figure -> {out_path}")

    # --- Fidelity metrics --------------------------------------------------------------------------
    print("\n[Validation] Computing fidelity metrics (final render's jewellery region vs. source asset)...")
    source_rgb_for_compare = jewellery_rgb  # the un-placed, un-warped source asset
    metrics = evaluate_fidelity(final_composite, source_rgb_for_compare, mask=placed_mask)
    delta_e = color_delta_e(final_composite, source_rgb_for_compare, mask=placed_mask)
    print_fidelity_report(metrics, delta_e)
else:
    print("\nNo composite produced — see the Stage 3 skip message above.")


## Summary of design decisions flagged in this notebook

- **SAM2 fine-tuning** is a documented plug-in point, not run here (no annotated jewellery dataset available).
- **ViTMatte** is tried first; a from-scratch guided-filter matte is a real (if weaker) fallback if the checkpoint can't be reached.
- **Landmark-based anchors** (earlobe, hairline parting, wrist width, finger width, ankle width) are documented, calibration-constant approximations — face/hand/pose models don't expose these measurements directly. Flagged per-category in Section 3.4's docstrings.
- **`CATEGORY_REFERENCE_MM`** (Section 4) is a placeholder population-average lookup, not a calibrated norms table.
- **`ai-toolkit`'s `model.arch`** value for FLUX.1-Fill and Qwen-Image-Edit LoRA training (Section 5.5/5.6) is a best-effort guess following neighbouring archs — flagged as needing verification against the current `ai-toolkit/config/examples/` before a real training run.
- **`color_delta_e`** uses `scikit-image` rather than `colormath`, per the build brief's own colormath-or-skimage allowance (colormath is effectively unmaintained).
- **SSIM/LPIPS target direction** (Section 6.2): the brief's literal "≥0.95 for both" doesn't work for LPIPS (a distance metric) — reinterpreted as an LPIPS ceiling of ≤0.05, flagged rather than silently applied.
- **Production acceptance rate ≥90%** is a business-process KPI needing a human-reviewed batch; `estimate_acceptance_rate()` is the aggregation stub for when that data exists.
- **Demo portrait** is a real sourced photo (Stage 2 needs photographic input); the **demo jewellery piece** is synthetic (Stage 1 segmentation of a simple shape doesn't need to be photorealistic to exercise the pipeline, but is not representative of real filigree performance).

**Not in scope here (per the brief):** deployment topology, model licensing resolution, hosting cost, and infrastructure.